# Chapter 5 &mdash; The Mod Algebra That Makes Residue States Work

**Concept 8 of the Chapter 5 decomposition:** *The Mod Algebra That Makes Residue States Computable*

$(a+b)\%N$ and $(ab)\%N$ can be computed from residues alone &mdash; which is exactly why finite memory suffices.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-Mod-Algebra/Concept-Mod-Algebra.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Residue states are only legitimate because of two identities:

$$(a+b)\bmod N = ((a\bmod N) + (b\bmod N))\bmod N$$
$$(a\cdot b)\bmod N = ((a\bmod N)\cdot(b\bmod N))\bmod N$$

They say the residue of a result depends **only on the residues of the inputs**, never
on the full values. That is a **homomorphism** from $\mathbb{Z}$ onto
$\mathbb{Z}_N$ &mdash; and it is precisely the property that lets a *finite* machine track
an *infinite* quantity.

When a condition has no such algebra (like "equal numbers of 0s and 1s"), no residue
trick exists, and the language turns out non-regular.

## 2. Definitions

### The two identities, as checkable predicates

In [ ]:
def add_ok(a, b, N): return (a + b) % N == ((a % N) + (b % N)) % N
def mul_ok(a, b, N): return (a * b) % N == ((a % N) * (b % N)) % N

### Why the DFA recurrence is a consequence

In [ ]:
def resid_from_scratch(s, m): return int(s, 2) % m if s else 0
def resid_incremental(s, m):
    r = 0
    for b in s: r = (2*r + int(b)) % m      # uses BOTH identities
    return r

<!-- nav-strip -->

---

&larr;&nbsp;[Ch5&nbsp;7.&nbsp;State Names as Residues: MSB-First "Divisible by 3"](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-Residue-States-MSB/Concept-Residue-States-MSB.ipynb) &nbsp;&middot;&nbsp; [**Chapter 5** index](https://github.com/ganeshutah/Jove/blob/master/Chapter5/README.md) &nbsp;&middot;&nbsp; [Ch5&nbsp;9.&nbsp;LSB-First "Divisible by 3": a Pair-Valued State](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-LSB-First-Pair-State/Concept-LSB-First-Pair-State.ipynb)&nbsp;&rarr;

---

## 3. Tests

The identities hold, exhaustively on a decent range.

In [ ]:
import random
assert all(add_ok(a, b, N) and mul_ok(a, b, N)
           for N in range(2, 12) for a in range(60) for b in range(60))
print("both identities verified for N in 2..11, a,b in 0..59")
for a, b, N in [(17, 25, 3), (1000, 999, 7)]:
    print("  (%d+%d)%%%d = %d = (%d+%d)%%%d" % (a,b,N,(a+b)%N,a%N,b%N,N))

Therefore the incremental residue equals the true residue &mdash; the DFA is justified.

In [ ]:
from itertools import product
for m in [3, 5, 7]:
    bad = [''.join(p) for k in range(1, 12) for p in product('01', repeat=k)
           if resid_incremental(''.join(p), m) != resid_from_scratch(''.join(p), m)]
    print("m=%d  mismatches: %d" % (m, len(bad)))
    assert not bad
print("\nThe state never holds N -- only N mod m -- and that is provably enough.")

Where the algebra runs out: a difference of counts has no bounded residue.

In [ ]:
def diff(s): return s.count('0') - s.count('1')
vals = {diff('0'*k + '1'*j) for k in range(12) for j in range(12)}
print("values the 'difference' quantity can take :", sorted(vals)[:8], "...")
print("unbounded -> no finite residue set -> no DFA.  (Chapter 4's L_01.)")

## 4. Exercises


1. Does the subtraction identity hold too? Check it for negative $a-b$.
2. Which algebraic property fails for "number of 0s minus number of 1s"?
3. State the homomorphism $\mathbb{Z}\to\mathbb{Z}_N$ precisely.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter5/Concept-Mod-Algebra')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')